# v2 게이팅 — 블렌드 증분 (채택 결정)

CatBoost 단일 +0.00033(게이트 통과)이 **운영 블렌드(3tree+lr+nn)로 전이되나?**가 진짜 채택 관문.
- base 3트리 vs **+v2게이팅** 3트리를 **동일 seed42 fold**로 재생성(게이팅 효과만 격리) → lr·nn(입력 CSV) 유지 → 힐클라이밍 → **(blend+v2 − blend) 페어드 부트스트랩 CI + 멀티시드.**
- 🔴 lr/nn 로드 버그 수정(라벨 'y' 차단)·없으면 하드스톱. 게이팅은 순수 행단위 산술(누수·fold-내부 불필요).
- **입력(Add Input):** train.csv/test.csv + `oof_predictions.csv`(lr) + `oof_nn.csv`. (v2 피처는 여기서 재생성.)

## 0. 준비 + tf_tree + v2 게이팅 + 3트리 인터페이스

In [1]:
import os,glob,re,time
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier
def find_csv(n):
    h=[p for p in glob.glob("/kaggle/input/**/*.csv",recursive=True) if os.path.basename(p)==n]; assert h,f"{n} 없음"; return sorted(h,key=len)[0]
train=pd.read_csv(find_csv("train.csv")); test=pd.read_csv(find_csv("test.csv"))
TARGET="임신 성공 여부"; ID_COL="ID"; y=train[TARGET].astype(int).values; N=len(train)
def NUM(df,c): return pd.to_numeric(df[c],errors="coerce") if c in df else pd.Series(np.nan,index=df.index)
def DIV(num,den): den=den.astype(float); return num.astype(float)/den.where(den>0)
def runr(x): return rankdata(x)/len(x)
COL_PROC="특정 시술 유형"; COL_RSN="배아 생성 주요 이유"
NOMINAL_COLS=["시술 시기 코드","시술 유형","배란 유도 유형","난자 출처","정자 출처"]
OCC=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수","총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
AGE_MAPS={"시술 당시 나이":{"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1},
 "난자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1},
 "정자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}}
CMAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
_tp=lambda s: [] if pd.isna(s) else [t.strip() for t in re.split(r"[/:]",str(s)) if t.strip()]
def fit_tree(tr):
    st={}; ig={TARGET,ID_COL}
    st["dead"]=[c for c in tr.columns if c not in ig and tr[c].nunique(dropna=True)<=1]
    st["sparse"]=[c for c in tr.columns if c not in ig and c not in st["dead"] and tr[c].isna().mean()>0.98]
    st["lc"]={c:pd.Index(tr[c].astype("category").cat.categories) for c in NOMINAL_COLS if c in tr}
    st["pv"]=sorted({t for L in tr[COL_PROC].apply(_tp) for t in L}); return st
def tf_tree(df,st):
    df=df.copy()
    if TARGET in df: df=df.drop(columns=[TARGET])
    df["is_DI"]=(df["시술 유형"]=="DI").astype(int)
    df=df.drop(columns=[c for c in st["dead"] if c in df.columns])
    for c in st["sparse"]:
        if c in df: df[f"{c}_있음"]=df[c].notna().astype(int); df=df.drop(columns=[c])
    for c in OCC:
        if c in df: df[c]=df[c].astype(object).map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].astype(object).map(m)
    cats=[]
    for c,cc in st["lc"].items():
        if c in df: df[c]=pd.Categorical(df[c],categories=cc).codes.astype("int32"); cats.append(c)
    ts=df[COL_PROC].apply(_tp)
    for v in st["pv"]: df[f"proc_{v}"]=ts.apply(lambda L,v=v:int(v in L))
    df=df.drop(columns=[c for c in [COL_PROC,COL_RSN,ID_COL] if c in df.columns])
    obj=[c for c in df.columns if df[c].dtype==object]
    if obj: df=df.drop(columns=obj)
    for c in cats: df[c]=df[c].fillna(-1).astype("int32")
    return df,[c for c in cats if c in df.columns]
# v2 게이팅 (순수 행단위 산술)
def masks(df):
    return {"신선":NUM(df,"신선 배아 사용 여부")==1,"동결":NUM(df,"동결 배아 사용 여부")==1,
            "ICSI":NUM(df,"미세주입된 난자 수")>0,
            "본인난자":df["난자 출처"].astype(str)=="본인 제공","기증난자":df["난자 출처"].astype(str)=="기증 제공"}
def build_v2_gated(df):
    M=masks(df); F={}
    P1=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")); P2=DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수"))
    P6=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")); L3=NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일")
    F["g신선_수정률"]=P1.where(M["신선"]); F["gICSI_수정효율"]=P2.where(M["ICSI"])
    F["g본인_난자수율"]=P6.where(M["본인난자"]); F["g기증_난자수율"]=P6.where(M["기증난자"]); F["g신선_배양일수"]=L3.where(M["신선"])
    F["FZ1_동결해동이식간격"]=(NUM(df,"배아 이식 경과일")-NUM(df,"배아 해동 경과일")).where(M["동결"])
    F["FZ2_해동이식률"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수")); F["FZ3_해동난자수율"]=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"해동 난자 수"))
    F["PG1_PGT강도"]=NUM(df,"착상 전 유전 검사 사용 여부").fillna(0)+NUM(df,"착상 전 유전 진단 사용 여부").fillna(0)
    F["PG2_PGT분류"]=NUM(df,"PGD 시술 여부").fillna(0)+NUM(df,"PGS 시술 여부").fillna(0)
    F["ST1_자극"]=NUM(df,"배란 자극 여부").fillna(0)
    return pd.DataFrame(F,index=df.index)
st=fit_tree(train); Xb,CATF=tf_tree(train,st); Xb_te,_=tf_tree(test,st); Xb_te=Xb_te.reindex(columns=Xb.columns)
V2tr=build_v2_gated(train); V2te=build_v2_gated(test)
Xv2=pd.concat([Xb.reset_index(drop=True),V2tr.reset_index(drop=True)],axis=1)
Xv2_te=pd.concat([Xb_te.reset_index(drop=True),V2te.reset_index(drop=True)],axis=1)
KINDS=["lgb","xgb","cat"]; KNAME={"lgb":"LightGBM","xgb":"XGBoost","cat":"CatBoost"}
LGP=dict(objective="binary",metric="auc",verbose=-1,learning_rate=0.05,num_leaves=63,feature_fraction=0.8,bagging_fraction=0.8,bagging_freq=1,min_child_samples=50)
XGP=dict(objective="binary:logistic",eval_metric="auc",tree_method="hist",learning_rate=0.05,max_depth=6,subsample=0.8,colsample_bytree=0.8)
def fit_one(kind,Xt,yt,Xv,yv,catf,seed):
    if kind=="lgb": return lgb.train(dict(LGP,seed=seed),lgb.Dataset(Xt,yt,categorical_feature=catf),1500,valid_sets=[lgb.Dataset(Xv,yv,categorical_feature=catf)],callbacks=[lgb.early_stopping(80,verbose=False),lgb.log_evaluation(0)])
    if kind=="xgb": return xgb.train(dict(XGP,seed=seed),xgb.DMatrix(Xt.values,label=yt),1500,evals=[(xgb.DMatrix(Xv.values,label=yv),"v")],early_stopping_rounds=80,verbose_eval=False)
    m=CatBoostClassifier(iterations=1500,learning_rate=0.05,depth=6,verbose=0,random_seed=seed,early_stopping_rounds=80); m.fit(Xt,yt,eval_set=(Xv,yv),cat_features=catf); return m
def pred_one(kind,m,X):
    if kind=="lgb": return m.predict(X)
    if kind=="xgb": return m.predict(xgb.DMatrix(X.values),iteration_range=(0,m.best_iteration+1))
    return m.predict_proba(X)[:,1]
print("base",Xb.shape,"| +v2게이팅",Xv2.shape,"| 3트리 준비")

base (256351, 72) | +v2게이팅 (256351, 83) | 3트리 준비


## 1. 【결정】 base 3트리 vs +v2게이팅 3트리 OOF (동일 seed42 fold)

In [2]:
def tree_oof(X,Xte,seed=42):
    folds=list(StratifiedKFold(5,shuffle=True,random_state=seed).split(X,y)); catf=[c for c in CATF if c in X.columns]
    out={}
    for kind in KINDS:
        oof=np.zeros(N); tst=np.zeros(len(Xte)); t0=time.time()
        for tri,vai in folds:
            m=fit_one(kind,X.iloc[tri],y[tri],X.iloc[vai],y[vai],catf,seed)
            oof[vai]=pred_one(kind,m,X.iloc[vai]); tst+=pred_one(kind,m,Xte)/len(folds)
        out[kind]=(oof,tst); print(f"  {KNAME[kind]:9s}: OOF={roc_auc_score(y,oof):.5f} ({time.time()-t0:.0f}s)")
    return out
print("base 3트리:"); base_trees=tree_oof(Xb,Xb_te,42)
print("v2게이팅 3트리:"); v2_trees=tree_oof(Xv2,Xv2_te,42)

base 3트리:
  LightGBM : OOF=0.73944 (31s)
  XGBoost  : OOF=0.73959 (52s)
  CatBoost : OOF=0.73963 (555s)
v2게이팅 3트리:
  LightGBM : OOF=0.73959 (31s)
  XGBoost  : OOF=0.73957 (50s)
  CatBoost : OOF=0.73970 (561s)


## 2. lr·nn OOF 로드 (버그수정·하드스톱)

In [3]:
def load_oof():
    lr=nn=None
    for p in glob.glob("/kaggle/input/**/oof_predictions.csv",recursive=True):
        d=pd.read_csv(p); lrk=[k for k in d.columns if "lr" in k.lower() and k.lower()!="y"]
        if lrk: lr=d[lrk[0]].values
        break
    for p in glob.glob("/kaggle/input/**/oof_nn.csv",recursive=True):
        d=pd.read_csv(p); nnk=[k for k in d.columns if k.lower()=="oof_nn"] or [k for k in d.columns if "nn" in k.lower()]
        nn=d[nnk[0]].values if nnk else None
        break
    return lr,nn
oof_lr,oof_nn=load_oof()
assert oof_lr is not None, "lr OOF 없음 — oof_predictions.csv Add Input 필수"
assert oof_nn is not None, "nn OOF 없음 — oof_nn.csv Add Input 필수"
assert roc_auc_score(y,oof_lr)<0.999 and roc_auc_score(y,oof_nn)<0.999, "라벨 잘못 로드 의심(AUC≈1.0)"
print(f"lr OOF={roc_auc_score(y,oof_lr):.5f} | nn OOF={roc_auc_score(y,oof_nn):.5f}")

lr OOF=0.71904 | nn OOF=0.73751


## 3. 힐클라이밍 블렌드 — base vs +v2 + 증분 CI

In [4]:
def hill(d,yy,n=120):
    nm=list(d); s0={k:roc_auc_score(yy,d[k]) for k in nm}; b=max(s0,key=s0.get); ens=[b]; s=d[b].copy(); best=(list(ens),s0[b])
    for _ in range(n):
        cb,ca=None,-1
        for k in nm:
            a=roc_auc_score(yy,(s+d[k])/(len(ens)+1))
            if a>ca: ca,cb=a,k
        ens.append(cb); s=s+d[cb]
        if ca>best[1]: best=(list(ens),ca)
    from collections import Counter; c=Counter(best[0]); return {k:c.get(k,0)/len(best[0]) for k in nm}, best[1]
def blend(trees):
    d={f"t_{k}":runr(trees[k][0]) for k in trees}; d["lr"]=runr(oof_lr); d["nn"]=runr(oof_nn)
    w,a=hill(d,y); return d,w,a
dB,wB,aB=blend(base_trees); dF,wF,aF=blend(v2_trees)
print(f"base 블렌드 OOF = {aB:.5f}  weights={ {k:round(v,3) for k,v in wB.items() if v>0} }")
print(f"+v2  블렌드 OOF = {aF:.5f}  weights={ {k:round(v,3) for k,v in wF.items() if v>0} }")
print(f"증분 Δ = {aF-aB:+.5f}")
pB=sum(wB[k]*dB[k] for k in wB); pF=sum(wF[k]*dF[k] for k in wF)
rng=np.random.default_rng(0); ds=[]
for _ in range(2000):
    ix=rng.integers(0,N,N); ds.append(roc_auc_score(y[ix],pF[ix])-roc_auc_score(y[ix],pB[ix]))
lo,hi=np.percentile(ds,[2.5,97.5]); gate=lo>0
print(f"\n블렌드 증분 = {aF-aB:+.5f} | 95% CI [{lo:+.5f},{hi:+.5f}]")
print("게이트(CI>0):","✅ 통과 — 멀티시드 확인 후 LB" if gate else "❌ 미달 — CatBoost 단일 이득이 블렌드로 전이 안 됨(lr/nn/기존멤버와 겹침)")

base 블렌드 OOF = 0.74045  weights={'t_lgb': 0.215, 't_xgb': 0.237, 't_cat': 0.29, 'lr': 0.022, 'nn': 0.237}
+v2  블렌드 OOF = 0.74055  weights={'t_lgb': 0.276, 't_xgb': 0.163, 't_cat': 0.306, 'lr': 0.02, 'nn': 0.235}
증분 Δ = +0.00010

블렌드 증분 = +0.00010 | 95% CI [-0.00005,+0.00026]
게이트(CI>0): ❌ 미달 — CatBoost 단일 이득이 블렌드로 전이 안 됨(lr/nn/기존멤버와 겹침)


## 4. 멀티시드 증분 확인 (게이트 통과 시) + 출력

In [5]:
import json
if not gate:
    print("게이트 미달 → 멀티시드 불필요. 공정한 네거티브로 종결(단일 CatBoost 이득은 블렌드에 흡수됨).")
    inc_ms=[aF-aB]
else:
    print("게이트 통과 → seed 7/2024 증분 확인:")
    inc_ms=[aF-aB]
    for sd in (7,2024):
        bt=tree_oof(Xb,Xb_te,sd); ft=tree_oof(Xv2,Xv2_te,sd)
        _,wb,ab=blend(bt); _,wf,af=blend(ft); inc_ms.append(af-ab)
        print(f"  seed{sd}: base={ab:.5f} +v2={af:.5f} 증분={af-ab:+.5f}")
    mu,sd_=np.mean(inc_ms),np.std(inc_ms)
    print(f"\n멀티시드 증분 평균={mu:+.5f} std={sd_:.5f}")
    print("최종:", "✅ 채택 — LB 1회 확인" if (mu>0 and abs(mu)>sd_) else "🟡 시드 불안정, 신중히")
# 출력: v2 블렌드 test 예측(트리부분) + OOF 저장
pd.DataFrame({"id":np.arange(N),**{f"v2_{k}":v2_trees[k][0] for k in v2_trees}}).to_csv("/kaggle/working/oof_v2_trees.csv",index=False)
pd.DataFrame({"ID":test[ID_COL] if ID_COL in test else np.arange(len(test)),**{f"v2_{k}":v2_trees[k][1] for k in v2_trees}}).to_csv("/kaggle/working/test_v2_trees.csv",index=False)
json.dump({"inc":aF-aB,"ci":[lo,hi],"gate":bool(gate),"inc_ms":inc_ms,"wF":{k:wF[k] for k in wF}},open("/kaggle/working/v2_blend_decision.json","w"),ensure_ascii=False)
print("\n💾 oof_v2_trees.csv / test_v2_trees.csv / v2_blend_decision.json 저장")
print("(최종 submission은 운영 블렌드 노트북에서 lr/nn test와 결합 — 여기선 트리부분·결정만.)")

게이트 미달 → 멀티시드 불필요. 공정한 네거티브로 종결(단일 CatBoost 이득은 블렌드에 흡수됨).

💾 oof_v2_trees.csv / test_v2_trees.csv / v2_blend_decision.json 저장
(최종 submission은 운영 블렌드 노트북에서 lr/nn test와 결합 — 여기선 트리부분·결정만.)
